# Classifier Generalization Comparison

This read-only notebook compares held-out binary performance, long-text performance, SemEval localization, and BBC safety diagnostics.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from notebooks.utils.classification_results_analysis_utils import document_metrics, load_classification_run
from notebooks.utils.classifier_generalization_comparison_utils import (
    held_out_binary_metrics, plot_dataset_heatmap, plot_f1_mae_frontier,
    plot_generalization_leaderboard, plot_parameter_effects, plot_safety_frontier,
    add_candidate_column, rank_generalization_candidates, select_comparison_metrics,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
RUN_ID = "feature-vector-classification-2026-09-15-rerun-2"
MIN_TEXT_TOKENS = 200
TOP_TIER_MACRO_F1_GAP = 0.02
results = load_classification_run(DATA_ROOT, run_id=RUN_ID)
metrics = document_metrics(results)
pd.set_option("display.max_columns", 50)

## Evidence And Cohorts

BBC is HG-only and is excluded from the primary binary ranking.

In [ ]:
held_out = held_out_binary_metrics(metrics)
if held_out.empty:
    display(Markdown("**insufficient_binary_holdout:** this run has no true unseen binary evaluation dataset."))
else:
    display(held_out.groupby(["training_regime", "dataset"], dropna=False).size().rename("rows").reset_index())
cohorts = {"All texts": select_comparison_metrics(metrics, min_text_tokens=None), f">= {MIN_TEXT_TOKENS}": select_comparison_metrics(metrics, min_text_tokens=MIN_TEXT_TOKENS)}
rankings = {}
for cohort_name, cohort_metrics in cohorts.items():
    rankings[cohort_name] = rank_generalization_candidates(add_candidate_column(cohort_metrics), pd.DataFrame(), top_tier_gap=TOP_TIER_MACRO_F1_GAP)
    display(rankings[cohort_name])

## Decision Figures

Each figure is rendered for both cohorts.

In [ ]:
for cohort_name, cohort_metrics in cohorts.items():
    ranking = rankings[cohort_name]
    display(plot_generalization_leaderboard(ranking, title=f"{cohort_name}: held-out leaderboard"))
    display(plot_dataset_heatmap(add_candidate_column(cohort_metrics), title=f"{cohort_name}: dataset macro-F1"))
    display(plot_f1_mae_frontier(ranking, title=f"{cohort_name}: macro-F1 and SemEval MAE"))
    for parameter in ("scorer_llm", "feature_set", "training_window", "evaluation_window"):
        if parameter in cohort_metrics.columns:
            display(plot_parameter_effects(cohort_metrics, parameter=parameter, title=f"{cohort_name}: {parameter}"))
    display(plot_safety_frontier(ranking, pd.DataFrame(), title=f"{cohort_name}: safety-aware frontier"))